# Full Pipeline: Petri Detector -> Colony Segmentation -> Anomaly Detection

Notebook-обёртка для `full_pipeline.py`.

Используются локальные модели из папки `full_pipline/models`:

- `petri_detector_yolo26s_best.pt`
- `colony_yolo26x_seg_best.pt`

По умолчанию обрабатываются 5 изображений из `C:/ColonyNet/Петри`.


In [ ]:
from pathlib import Path
import sys

PIPELINE_DIR = Path.cwd()
if PIPELINE_DIR.name != "full_pipline":
    PIPELINE_DIR = Path("C:/ColonyNet/full_pipline")

PROJECT_ROOT = PIPELINE_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PIPELINE_DIR:", PIPELINE_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
from full_pipline.full_pipeline import (
    DEFAULT_PETRI_IMAGES,
    MODELS_DIR,
    make_full_pipeline_config,
    run_full_pipeline_for_images,
)

config = make_full_pipeline_config()
config

In [ ]:
print("Models dir:", MODELS_DIR)
print("Petri detector:", config.petri_detector_weights_path, Path(config.petri_detector_weights_path).exists())
print("Colony segmenter:", config.model_weights_path, Path(config.model_weights_path).exists())
print("Output dir:", config.output_dir)

In [ ]:
for image_path in DEFAULT_PETRI_IMAGES:
    print(image_path, image_path.exists())

## Запуск полного прогона

Ячейка ниже выполняет полный pipeline для всех 5 изображений:

1. детекция чашки Петри;
2. crop/resize до 736x736;
3. YOLO26x-seg instance segmentation;
4. извлечение признаков;
5. расчёт anomaly score;
6. сохранение CSV/XLSX/PNG.


In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    full_result = run_full_pipeline_for_images(config=config)
    display(full_result["summary"])
    display(full_result["selected"].head())
    display(full_result["review_candidates"].head())
    display(full_result["technical_warnings"].head())
else:
    print("Для запуска полного прогона установите RUN_FULL_PIPELINE = True")


## Выходные файлы

После запуска результаты будут сохранены в `outputs/colony_anomaly_detection/petri_full_pipeline`.

Основные файлы:

- `petri_5_images_summary.csv`
- `petri_5_images_selected_anomalies.csv`
- `petri_5_images_review_candidates.csv`
- `petri_5_images_technical_warnings.csv`
- `petri_5_images_report.xlsx`

Для каждого изображения также создаётся отдельная подпапка с визуализацией, масками и подробными таблицами.
